# Versão 2: Pipeline Full Python com Scikit-Learn (Tratamento + Modelos + Métricas)
**Equipe 11:** Guilherme Santos Paz e Ana Melina  
**Base de Dados:** Nº 22 - Default of Credit Card Clients (ID UCI: 350)  
**Protocolo:** Validação Cruzada Estratificada de 10 Partições (`10-fold`) com Semente Fixa (`seed = 42`).  
**Modelos:** Baseline (`DummyClassifier` = `ZeroR`), Categoria *Functions* (`LogisticRegression` = `Logistic`) e Categoria *Rules* (`RIPPER` / Regras de Decisão).

In [1]:
# CÉLULA 1: Importação das bibliotecas
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

In [2]:
# CÉLULA 2: Carregamento da base, limpeza estrutural e amostragem estratificada (seed=42)
df = pd.read_excel('default of credit card clients.xls', header=1)

# Remoção de ID e renomeação da classe
df = df.drop(columns=['ID'])
df = df.rename(columns={'default payment next month': 'CLASSE_INADIMPLENCIA'})

# Tratamento dos códigos não documentados em EDUCATION, MARRIAGE e PAY_0..PAY_6
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

colunas_pay = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for col in colunas_pay:
    df[col] = df[col].replace([-2, -1], 0)

# Amostragem estratificada para 5.000 instâncias com seed=42
df_amostra, _ = train_test_split(
    df,
    train_size=5000,
    stratify=df['CLASSE_INADIMPLENCIA'],
    random_state=42
)
df_amostra = df_amostra.reset_index(drop=True)
print('Amostra pronta:', df_amostra.shape)

Amostra pronta: (5000, 24)


In [3]:
# CÉLULA 3: Preparação dos Dados para a EXECUÇÃO 1 (Atributos Contínuos Originais + Categóricos)
colunas_categoricas = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
colunas_continuas = [
    'LIMIT_BAL', 'AGE',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

X_exec1_bruto = df_amostra.drop(columns=['CLASSE_INADIMPLENCIA'])
y = df_amostra['CLASSE_INADIMPLENCIA'].values

# Padroniza as contínuas e aplica One-Hot Encoding nas categóricas
transformador_exec1 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), colunas_continuas),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), colunas_categoricas)
])

X_exec1 = transformador_exec1.fit_transform(X_exec1_bruto)
print('Matriz X_exec1 pronta:', X_exec1.shape)

Matriz X_exec1 pronta: (5000, 58)


In [4]:
# CÉLULA 4: Preparação dos Dados para a EXECUÇÃO 2 (Todos os Atributos Discretizados)
df_disc = df_amostra.copy()

df_disc['AGE'] = pd.cut(df_disc['AGE'], bins=[0, 29, 45, 60, 120], labels=['Jovem', 'Adulto', 'Meia_Idade', 'Senior']).astype(str)
df_disc['LIMIT_BAL'] = pd.qcut(df_disc['LIMIT_BAL'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop').astype(str)

for col in ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']:
    df_disc[col] = pd.cut(df_disc[col], bins=[-np.inf, 0, 10000, 50000, np.inf], labels=['Zero_Neg', 'Ate_10k', 'De_10k_50k', 'Acima_50k']).astype(str)

for col in ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']:
    df_disc[col] = pd.cut(df_disc[col], bins=[-np.inf, 0, 2000, 10000, np.inf], labels=['Zero', 'Ate_2k', 'De_2k_10k', 'Acima_10k']).astype(str)

X_exec2_bruto = df_disc.drop(columns=['CLASSE_INADIMPLENCIA']).astype(str)

# Na Execução 2 todos os 23 atributos são categóricos/discretizados
codificador_exec2 = OneHotEncoder(drop='first', sparse_output=False)
X_exec2 = codificador_exec2.fit_transform(X_exec2_bruto)
print('Matriz X_exec2 (100% discretizada) pronta:', X_exec2.shape)

Matriz X_exec2 (100% discretizada) pronta: (5000, 86)


In [5]:
# CÉLULA 5: Definição do Protocolo 10-Fold Estratificado (seed=42) e Execução do Baseline (ZeroR)
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

modelo_zeror = DummyClassifier(strategy='most_frequent', random_state=42)
pred_zeror = cross_val_predict(modelo_zeror, X_exec1, y, cv=kfold)

print('=== RESULTADOS BASELINE: ZEROR (DummyClassifier) ===')
print('Acurácia:', round(accuracy_score(y, pred_zeror) * 100, 2), '%')
print('Estatística Kappa:', round(cohen_kappa_score(y, pred_zeror), 4))
print('Matriz de Confusão:\n', confusion_matrix(y, pred_zeror))
print('\nMétricas por Classe (0 = Adimplente, 1 = Inadimplente):')
print(classification_report(y, pred_zeror, target_names=['Adimplente (0)', 'Inadimplente (1)'], zero_division=0))

=== RESULTADOS BASELINE: ZEROR (DummyClassifier) ===
Acurácia: 77.88 %
Estatística Kappa: 0.0
Matriz de Confusão:
 [[3894    0]
 [1106    0]]

Métricas por Classe (0 = Adimplente, 1 = Inadimplente):
                  precision    recall  f1-score   support

  Adimplente (0)       0.78      1.00      0.88      3894
Inadimplente (1)       0.00      0.00      0.00      1106

        accuracy                           0.78      5000
       macro avg       0.39      0.50      0.44      5000
    weighted avg       0.61      0.78      0.68      5000



In [6]:
# CÉLULA 6: Categoria FUNCTIONS (Regressão Logística) - Execução 1 (Contínua) vs Execução 2 (Discretizada)
modelo_logistic = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)

# Execução 1: Base Contínua
pred_log_exec1 = cross_val_predict(modelo_logistic, X_exec1, y, cv=kfold)
print('=== CATEGORIA FUNCTIONS: LOGISTIC (EXECUÇÃO 1 - CONTÍNUA) ===')
print('Acurácia:', round(accuracy_score(y, pred_log_exec1) * 100, 2), '%')
print('Estatística Kappa:', round(cohen_kappa_score(y, pred_log_exec1), 4))
print('Matriz de Confusão:\n', confusion_matrix(y, pred_log_exec1))
print(classification_report(y, pred_log_exec1, target_names=['Adimplente (0)', 'Inadimplente (1)']))

# Execução 2: Base Discretizada
pred_log_exec2 = cross_val_predict(modelo_logistic, X_exec2, y, cv=kfold)
print('=== CATEGORIA FUNCTIONS: LOGISTIC (EXECUÇÃO 2 - DISCRETIZADA) ===')
print('Acurácia:', round(accuracy_score(y, pred_log_exec2) * 100, 2), '%')
print('Estatística Kappa:', round(cohen_kappa_score(y, pred_log_exec2), 4))
print('Matriz de Confusão:\n', confusion_matrix(y, pred_log_exec2))
print(classification_report(y, pred_log_exec2, target_names=['Adimplente (0)', 'Inadimplente (1)']))

c:\Users\guilh\OneDrive\Área de Trabalho\Projetos\MineraçãodeDados\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\guilh\OneDrive\Área de Trabalho\Projetos\MineraçãodeDados\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instea

=== CATEGORIA FUNCTIONS: LOGISTIC (EXECUÇÃO 1 - CONTÍNUA) ===
Acurácia: 81.72 %
Estatística Kappa: 0.3596
Matriz de Confusão:
 [[3702  192]
 [ 722  384]]
                  precision    recall  f1-score   support

  Adimplente (0)       0.84      0.95      0.89      3894
Inadimplente (1)       0.67      0.35      0.46      1106

        accuracy                           0.82      5000
       macro avg       0.75      0.65      0.67      5000
    weighted avg       0.80      0.82      0.79      5000



c:\Users\guilh\OneDrive\Área de Trabalho\Projetos\MineraçãodeDados\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\guilh\OneDrive\Área de Trabalho\Projetos\MineraçãodeDados\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instea

=== CATEGORIA FUNCTIONS: LOGISTIC (EXECUÇÃO 2 - DISCRETIZADA) ===
Acurácia: 81.7 %
Estatística Kappa: 0.3551
Matriz de Confusão:
 [[3709  185]
 [ 730  376]]
                  precision    recall  f1-score   support

  Adimplente (0)       0.84      0.95      0.89      3894
Inadimplente (1)       0.67      0.34      0.45      1106

        accuracy                           0.82      5000
       macro avg       0.75      0.65      0.67      5000
    weighted avg       0.80      0.82      0.79      5000



c:\Users\guilh\OneDrive\Área de Trabalho\Projetos\MineraçãodeDados\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [7]:
# CÉLULA 7: Categoria RULES (Algoritmo RIPPER / JRip via wittgenstein) - Execução 1 vs Execução 2
# Obs.: Instalar pacote antes no terminal se necessário: pip install wittgenstein
import wittgenstein as lw

# Execução 1 (Atributos Contínuos Originais)
df_ripper_exec1 = df_amostra.copy()
pred_ripper_exec1 = np.zeros(len(df_ripper_exec1), dtype=int)

for idx_treino, idx_teste in kfold.split(df_ripper_exec1, y):
    treino_fold = df_ripper_exec1.iloc[idx_treino]
    teste_fold = df_ripper_exec1.iloc[idx_teste].drop(columns=['CLASSE_INADIMPLENCIA'])
    
    modelo_ripper = lw.RIPPER(random_state=42, prune_size=0.33, k=2)
    modelo_ripper.fit(treino_fold, class_feat='CLASSE_INADIMPLENCIA', pos_class=1)
    pred_ripper_exec1[idx_teste] = modelo_ripper.predict(teste_fold)

print('=== CATEGORIA RULES: RIPPER / JRip (EXECUÇÃO 1 - CONTÍNUA) ===')
print('Acurácia:', round(accuracy_score(y, pred_ripper_exec1) * 100, 2), '%')
print('Estatística Kappa:', round(cohen_kappa_score(y, pred_ripper_exec1), 4))
print('Matriz de Confusão:\n', confusion_matrix(y, pred_ripper_exec1))
print(classification_report(y, pred_ripper_exec1, target_names=['Adimplente (0)', 'Inadimplente (1)']))

# Execução 2 (Atributos Discretizados)
df_ripper_exec2 = df_disc.copy()
pred_ripper_exec2 = np.zeros(len(df_ripper_exec2), dtype=int)

for idx_treino, idx_teste in kfold.split(df_ripper_exec2, y):
    treino_fold = df_ripper_exec2.iloc[idx_treino]
    teste_fold = df_ripper_exec2.iloc[idx_teste].drop(columns=['CLASSE_INADIMPLENCIA'])
    
    modelo_ripper_disc = lw.RIPPER(random_state=42, prune_size=0.33, k=2)
    modelo_ripper_disc.fit(treino_fold, class_feat='CLASSE_INADIMPLENCIA', pos_class=1)
    pred_ripper_exec2[idx_teste] = modelo_ripper_disc.predict(teste_fold)

print('=== CATEGORIA RULES: RIPPER / JRip (EXECUÇÃO 2 - DISCRETIZADA) ===')
print('Acurácia:', round(accuracy_score(y, pred_ripper_exec2) * 100, 2), '%')
print('Estatística Kappa:', round(cohen_kappa_score(y, pred_ripper_exec2), 4))
print('Matriz de Confusão:\n', confusion_matrix(y, pred_ripper_exec2))
print(classification_report(y, pred_ripper_exec2, target_names=['Adimplente (0)', 'Inadimplente (1)']))

# Exibe o conjunto final de regras geradas pelo RIPPER para colocar no relatório
print('\nRegras Lógicas Extraídas (RIPPER):')
modelo_ripper_disc.out_model()

=== CATEGORIA RULES: RIPPER / JRip (EXECUÇÃO 1 - CONTÍNUA) ===
Acurácia: 81.0 %
Estatística Kappa: 0.3145
Matriz de Confusão:
 [[3721  173]
 [ 777  329]]
                  precision    recall  f1-score   support

  Adimplente (0)       0.83      0.96      0.89      3894
Inadimplente (1)       0.66      0.30      0.41      1106

        accuracy                           0.81      5000
       macro avg       0.74      0.63      0.65      5000
    weighted avg       0.79      0.81      0.78      5000

=== CATEGORIA RULES: RIPPER / JRip (EXECUÇÃO 2 - DISCRETIZADA) ===
Acurácia: 80.84 %
Estatística Kappa: 0.3048
Matriz de Confusão:
 [[3724  170]
 [ 788  318]]
                  precision    recall  f1-score   support

  Adimplente (0)       0.83      0.96      0.89      3894
Inadimplente (1)       0.65      0.29      0.40      1106

        accuracy                           0.81      5000
       macro avg       0.74      0.62      0.64      5000
    weighted avg       0.79      0.81      0